In [25]:
import numpy as np

# 1. 定义基础 Reber 语法的路线图 (状态机)
# 字典的 key 是当前节点，value 是 [(下一个节点, 输出字母), ...]
REBER_GRAMMAR = {
    0: [(1, 'B')],
    1: [(2, 'T'), (3, 'P')],
    2: [(2, 'S'), (4, 'X')],
    3: [(3, 'T'), (5, 'V')],
    4: [(3, 'X'), (6, 'S')],
    5: [(4, 'P'), (6, 'V')],
    6: [(7, 'E')]
}

# 字母表，用于造假数据
VOCAB = "BPTXVSE"

# 制造一条【标准的】基础 Reber 字符串
def generate_standard_reber():
    state = 0
    string = ""
    while state != 7:
        transitions = REBER_GRAMMAR[state]
        # 遇到分叉路口，抛硬币随机选一条
        idx = np.random.randint(len(transitions))
        state, char = transitions[idx]
        string += char
    return string

# 制造一条【真的】嵌入式 Reber 字符串 (标签为 1)
def generate_valid_embedded_reber():
    std_reber = generate_standard_reber()
    # 随机选择是用 T 包裹还是用 P 包裹
    if np.random.rand() < 0.5:
        return f"BT{std_reber}TE"
    else:
        return f"BP{std_reber}PE"

# 制造一条【假的】嵌入式 Reber 字符串 (标签为 0)
def generate_invalid_embedded_reber():
    valid_string = list(generate_valid_embedded_reber())
    # 搞破坏的战术：随机挑一个位置，换成一个错误的字母
    idx = np.random.randint(0, len(valid_string))
    original_char = valid_string[idx]
    
    # 找一个跟原来不一样的字母替换上去
    available_wrong_chars = list(set(VOCAB) - set(original_char))
    valid_string[idx] = np.random.choice(available_wrong_chars)
    
    return "".join(valid_string)

In [29]:
import tensorflow as tf
import random
# 建立密码本：把字母变成数字 ID
char_to_id = {char: idx + 1 for idx, char in enumerate(VOCAB)}
# 注意：ID 0 留给 Padding (补齐用的空气)
random.seed(42)
def generate_dataset(size):
    """生成最终的训练数据集，一半真，一半假"""
    good_strings = [generate_valid_embedded_reber() for _ in range(size // 2)]
    bad_strings = [generate_invalid_embedded_reber() for _ in range(size // 2)]
    
    all_strings = good_strings + bad_strings
    # 标签：前一半是真的 (1)，后一半是假的 (0)
    labels = [1] * (size // 2) + [0] * (size // 2)
    
    # 查字典：把字符串变成数字 ID 列表
    X_ids = [[char_to_id[c] for c in string] for string in all_strings]
    
    # 补齐长度：因为句子长短不一，我们在后面补 0 (padding)
    X_padded = tf.keras.preprocessing.sequence.pad_sequences(X_ids, padding='post')
    
    # 打乱顺序，防止模型摸清规律
    indices = np.random.permutation(size)
    X_batch = X_padded[indices]
    y_batch = np.array(labels)[indices]
    
    return X_batch, y_batch

# ----------------- 战术测试 -----------------
X_train, y_train = generate_dataset(1000) # 生成 1000 条训练数据
print(f"X_train 形状 (批次, 最大长度): {X_train.shape}")
print(f"y_train 形状 (批次, 标签): {y_train.shape}")
print(f"第一条数据长这样: {X_train[0]} (真实标签: {y_train[0]})")

X_train 形状 (批次, 最大长度): (1000, 34)
y_train 形状 (批次, 标签): (1000,)
第一条数据长这样: [1 3 1 2 5 2 4 5 2 6 7 3 4 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0] (真实标签: 0)


In [35]:
np.random.seed(42)
tf.random.set_seed(42)

embedding_size = 5

model = tf.keras.Sequential([
    tf.keras.layers.InputLayer(input_shape=[None], dtype=tf.int32, ragged=True),
    tf.keras.layers.Embedding(input_dim=len(VOCAB)+1,#这个+1 就是因为有0，如果没有X_padded这一步，就不需要+1
                              output_dim=embedding_size,
                             mask_zero=True),#有0，但是这些数据是没有用的，所以需要设置mask_zero = True,来设置model不去看这些0.
    tf.keras.layers.GRU(30),
    tf.keras.layers.Dense(1, activation="sigmoid")
])
optimizer = tf.keras.optimizers.SGD(learning_rate=0.01, momentum = 0.95,
                                    nesterov=True)#学习率设置的很小，学习的比较慢，但是每一轮很快，就可以设置很多的轮次epochs。
model.compile(loss="binary_crossentropy", optimizer=optimizer,
              metrics=["accuracy"])
history = model.fit(X_train, y_train, epochs=300,
                    )

Epoch 1/300
32/32 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step - accuracy: 0.5070 - loss: 0.6931
Epoch 2/300
32/32 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.5210 - loss: 0.6930
Epoch 3/300
32/32 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.5280 - loss: 0.6924
Epoch 4/300
32/32 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.5310 - loss: 0.6919
Epoch 5/300
32/32 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.5340 - loss: 0.6911
Epoch 6/300
32/32 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.5280 - loss: 0.6902
Epoch 7/300
32/32 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.5500 - loss: 0.6890
Epoch 8/300
32/32 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.5690 - loss: 0.6875
Epoch 9/300
32/32 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.5690 - loss: 0.6856
Epoch 10/300
32/32 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.5730 - loss: 0.6833
Epoch 11/300
32/32 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.5680 - loss: 0.6808
Epoch 12/300
32/32 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy:

In [41]:
X_test,y_test = generate_dataset(1000)
loss, accuracy = model.evaluate(X_test, y_test)

32/32 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - accuracy: 1.0000 - loss: 1.8837e-04


In [44]:
X_test,y_test = generate_dataset(1000)
loss, accuracy = model.evaluate(X_test, y_test)

32/32 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - accuracy: 0.9840 - loss: 0.1281    


In [45]:
X_test,y_test = generate_dataset(1000)
loss, accuracy = model.evaluate(X_test, y_test)

32/32 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - accuracy: 0.9930 - loss: 0.0470
